# Andean Obsidian Geochemical Sourcing

Welcome to the geochemical analysis notebook for XRF sourcing of obsidian from the Andes of South America.
This notebook provides the code for displaying geochemistry data together with obsidian source data from the region. It is intended to be reproducible and to provide a model for use with other data or analyses.

For long-term archival use, this notebook is paired with dependency files (`environment.yml`, `requirements.txt`) in the repository root that specify minimum compatible versions, so a working Python environment can be resolved in the future on whatever kernel/OS is available (local Miniconda, WSL, Binder, etc.).

Run all the cells before uploading files or clicking any buttons — use the menu **Run > Run All Cells** (in Voilà/Binder this happens automatically). The three tabs below walk through the workflow in order:

1. **Data Upload** — provide the three source tables (upload CSVs, or point to a Google Sheet / direct CSV URL).
2. **Map Selection** — the source chemistry and source-location tables are joined by `Group`, and you lasso-select obsidian sources of interest on a map.
3. **Biplot & Ternary** — biplot and ternary plots update automatically from your map selection so you can compare study samples against known obsidian sources.

The Python can be modified and re-run, and maps and plots are interactive and can be saved.


In [ ]:
# CELL 1 — Imports
# Environment bootstrap for local, Jupyter, and Voila usage.
# For archival reproducibility, prefer the environment.yml / requirements.txt
# files in the repository root (minimum-version constraints, not exact pins,
# so they resolve on whatever kernel/OS is available).

from pathlib import Path
import os
import re
import sys
import io
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from types import SimpleNamespace

_is_binder = bool(os.environ.get("BINDER_LAUNCH_HOST"))
_is_voila = os.environ.get("SERVER_SOFTWARE", "").startswith("voila")

# Only attempt an on-the-fly pip install when running as a plain local kernel
# (not Binder/Voila, which already build their environment from environment.yml
# / requirements.txt ahead of time) AND only if the required packages are not
# already importable. This avoids re-installing/rebuilding packages (e.g.
# compiling numpy from source when no compiler is present) every time the
# notebook is run inside an already-working environment.
def _required_packages_importable():
    try:
        import numpy, pandas, plotly, ipywidgets  # noqa: F401
        return True
    except ImportError:
        return False

if not _is_binder and not _is_voila:
    if _required_packages_importable():
        print("Required packages already importable; skipping pip install.")
        print("If you have not done so, create the environment with:")
        print("  conda env create -f environment.yml   (conda/miniconda)")
        print("  python -m pip install -r requirements.txt   (pip/venv)")
    else:
        req_path = Path.cwd() / "requirements.txt"
        if not req_path.exists():
            req_path = Path.cwd().parent / "requirements.txt"

        if req_path.exists():
            print(f"Using dependency file: {req_path}")
            print("Installing dependencies from requirements.txt...")
            try:
                %pip install -q -r "{req_path}"
            except Exception as exc:
                print(f"⚠️ Automatic pip install failed: {exc}")
                print("Create the environment manually instead:")
                print("  conda env create -f environment.yml   (conda/miniconda, recommended)")
                print("  python -m pip install -r requirements.txt   (pip/venv)")
        else:
            print("No requirements.txt found. Create the environment with: conda env create -f environment.yml")
elif _is_voila:
    print("Voila-managed environment detected; dependencies come from the pre-built environment.")
else:
    print("Binder-managed environment detected; dependencies come from environment.yml")



try:
    from plotly.graph_objects import FigureWidget
    _plotly_figure_widget_available = True
except Exception:
    FigureWidget = go.Figure
    _plotly_figure_widget_available = False

from plotly.colors import DEFAULT_PLOTLY_COLORS
from ipywidgets import Button, Output, VBox, HBox, Tab, SelectMultiple, Layout, FileUpload, Text
import ipywidgets as widgets
from IPython.display import display, HTML

# Detect environment
def detect_environment():
    if os.environ.get("BINDER_LAUNCH_HOST"):
        return "binder"
    if os.environ.get("SERVER_SOFTWARE", "").startswith("voila"):
        return "voila"
    if os.environ.get("VSCODE_PID") or os.environ.get("TERM_PROGRAM") == "vscode":
        return "vscode"
    return "jupyter"

ENV = detect_environment()
print(f"Environment : {ENV}")
print(f"Python      : {sys.version.split()[0]}")
print(f"FigureWidget: {'available' if _plotly_figure_widget_available else 'falling back to go.Figure'}")

# Scrollable output — only inject in environments that support it
if ENV in ("jupyter", "vscode", "binder", "voila"):
    display(HTML("""
        <style>
            .output_wrapper, .output { 
                max-height: 600px !important; 
                overflow-y: auto !important; 
            }
        </style>
    """))

# ── Shared state object ────────────────────────────────────────────────────
# SimpleNamespace allows attribute access (state.srcs).
# Every step below reads and writes to this object so data flows between
# the tabs without relying on manually re-running cells, which is required
# for unattended execution under Voila/Binder.
state = SimpleNamespace(
    srcs=pd.DataFrame(),          # cleaned source chemistry
    srcs_locs=pd.DataFrame(),     # source coordinates
    study=pd.DataFrame(),         # cleaned study sample chemistry
    srcs_subset=pd.DataFrame(),   # sources filtered to the map selection (>=2 samples)
    onesample=pd.DataFrame(),     # selected sources with < 2 samples
    selected_names=[],
    selected_groups=[],
    name_to_color={},
)

# Raw (pre-cleaning) dataframes, populated from default local files, uploads,
# or URLs (see Tab 1 below), and only cleaned/joined once "Load & Continue" runs.
_raw = SimpleNamespace(srcs=pd.DataFrame(), srcs_locs=pd.DataFrame(), study=pd.DataFrame())


In [ ]:
# CELL 2 - DATA LOADING HELPERS
# Pure helper functions used by Tab 1 (Data Upload). Nothing executes or
# displays here — the tab is built later once every function is defined,
# which keeps this notebook safe for unattended Voila/Binder execution.

# Default local CSV files (relative to the notebook or its parent folder / data subfolder)
DEFAULT_SRC_CHEM_FILE = "South_Am_sources_southern_XRF.csv"
DEFAULT_SRC_COORD_FILE = "South_Am_sources_all_locations.csv"
DEFAULT_STUDY_FILE = "study_samples.csv"


def read_csv_with_encodings(content):
    """Read CSV content (a path, URL, or in-memory content) with multiple encoding attempts."""
    if isinstance(content, (str, Path)) and not str(content).startswith(("http://", "https://")) and Path(content).exists():
        for encoding in ("utf-8", "latin1", "cp1252"):
            try:
                return pd.read_csv(content, encoding=encoding)
            except UnicodeDecodeError:
                continue
        return pd.read_csv(content, encoding="latin1")
    if isinstance(content, str) and content.startswith(("http://", "https://")):
        # URL: let pandas fetch it directly (works for direct CSV links and
        # Google Sheets "export?format=csv" links produced by gsheet_to_csv_url).
        return pd.read_csv(content)
    for encoding in ("utf-8", "latin1", "cp1252"):
        try:
            if isinstance(content, str):
                return pd.read_csv(io.StringIO(content), encoding=encoding)
            else:
                return pd.read_csv(io.BytesIO(content), encoding=encoding)
        except UnicodeDecodeError:
            continue
    if isinstance(content, str):
        return pd.read_csv(io.StringIO(content), encoding="latin1")
    else:
        return pd.read_csv(io.BytesIO(content), encoding="latin1")


def gsheet_to_csv_url(url):
    """Convert a typical Google Sheets share URL into a direct CSV export URL.

    Accepts links like
    ``https://docs.google.com/spreadsheets/d/<ID>/edit#gid=<GID>`` and returns
    ``https://docs.google.com/spreadsheets/d/<ID>/export?format=csv&gid=<GID>``.
    Any URL that doesn't match the Google Sheets pattern is returned unchanged
    (so plain direct-CSV URLs also work).
    """
    url = (url or "").strip()
    if not url:
        return url
    match = re.match(r"https://docs\.google\.com/spreadsheets/d/([a-zA-Z0-9-_]+)", url)
    if not match:
        return url
    sheet_id = match.group(1)
    gid_match = re.search(r"[?&#]gid=(\d+)", url)
    gid = gid_match.group(1) if gid_match else "0"
    return f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"


def _first_upload(upload_widget):
    val = upload_widget.value
    if isinstance(val, dict):
        if not val:
            return None
        return next(iter(val.values()))
    if isinstance(val, (list, tuple)):
        if not val:
            return None
        return val[0]
    return None


def _find_default_file(filename):
    """Search a small set of likely locations for a bundled default CSV."""
    if not filename:
        return None
    search_dirs = [
        Path.cwd(),
        Path.cwd() / "data",
        Path.cwd().parent,
        Path.cwd().parent / "data",
    ]
    for base_dir in search_dirs:
        path = base_dir / filename
        if path.exists():
            return path
    return None


def _clean_locs(raw_locs):
    """Normalize location and schema fields for a source-locations table."""
    rename_map = {
        "Chem_Group": "Group",
        "Latitude": "Lat",
        "Longitude": "Long",
        "Source": "Name"
    }
    raw_locs = raw_locs.rename(columns=rename_map)
    if "Name" not in raw_locs.columns and "Group" in raw_locs.columns:
        raw_locs["Name"] = raw_locs["Group"].astype("string")
    if "Group" in raw_locs.columns:
        raw_locs["Group"] = raw_locs["Group"].astype("string").str.strip()
    for col in ["Lat", "Long"]:
        if col in raw_locs.columns:
            raw_locs[col] = pd.to_numeric(raw_locs[col], errors="coerce")
    return raw_locs.dropna(subset=[col for col in ["Lat", "Long"] if col in raw_locs.columns]).reset_index(drop=True)


def _clean_study(raw_study):
    """Align study-sample column names to the schema used by this notebook."""
    raw_study = raw_study.rename(columns={"Name": "Sample", "Application": "Group"})
    if "Group" not in raw_study.columns and "Application" in raw_study.columns:
        raw_study = raw_study.rename(columns={"Application": "Group"})
    if "Sample" not in raw_study.columns and "Name" in raw_study.columns:
        raw_study = raw_study.rename(columns={"Name": "Sample"})
    return raw_study


def load_default_local_files():
    """Attempt to load the three bundled default CSVs into _raw (silent on failure)."""
    messages = []

    default_chem_path = _find_default_file(DEFAULT_SRC_CHEM_FILE)
    if default_chem_path is not None:
        try:
            _raw.srcs = read_csv_with_encodings(default_chem_path)
            messages.append(f"✅ Loaded default Sources Chemistry: {default_chem_path} ({len(_raw.srcs)} rows, {len(_raw.srcs.columns)} cols)")
        except Exception as e:
            messages.append(f"❌ Could not load default Sources Chemistry ({default_chem_path}): {e}")
    else:
        messages.append(f"ℹ️ Default Sources Chemistry file '{DEFAULT_SRC_CHEM_FILE}' not found; upload one or provide a URL below.")

    default_loc_path = _find_default_file(DEFAULT_SRC_COORD_FILE)
    if default_loc_path is not None:
        try:
            _raw.srcs_locs = _clean_locs(read_csv_with_encodings(default_loc_path))
            messages.append(f"✅ Loaded default Sources Locations: {default_loc_path} ({len(_raw.srcs_locs)} rows)")
        except Exception as e:
            messages.append(f"❌ Could not load default Sources Locations ({default_loc_path}): {e}")
    else:
        messages.append(f"ℹ️ Default Sources Locations file '{DEFAULT_SRC_COORD_FILE}' not found; upload one or provide a URL below.")

    default_study_path = _find_default_file(DEFAULT_STUDY_FILE)
    if default_study_path is not None:
        try:
            _raw.study = _clean_study(read_csv_with_encodings(default_study_path))
            messages.append(f"✅ Loaded default Study Samples: {default_study_path} ({len(_raw.study)} rows)")
        except Exception as e:
            messages.append(f"❌ Could not load default Study Samples ({default_study_path}): {e}")
    else:
        messages.append(f"ℹ️ Default Study Samples file '{DEFAULT_STUDY_FILE}' not found; upload one or provide a URL below.")

    return messages


In [ ]:
# CELL 3 - DATA CLEANING, VALIDATION AND JOIN
# Pure functions, called once the user clicks "Load & Continue" on Tab 1.

def clean_geochem_df(df):
    """Clean geochemistry dataframe headers and types."""
    if df is None or df.empty:
        return df

    # Make a copy to avoid SettingWithCopyWarning
    df = df.copy()

    # Remove Bruker artifacts and spaces
    df.columns = df.columns.str.replace(r'(Ka1|La1|\s+)', '', regex=True)

    # String columns - assign column by column to avoid pandas multi-column assignment ValueError
    string_cols = ['Group', 'Sample', 'Name']
    for c in string_cols:
        if c in df.columns:
            df[c] = df[c].astype('string')

    # Numeric columns
    present_strings = [c for c in string_cols if c in df.columns]
    numeric_cols = [c for c in df.columns if c not in present_strings]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # Drop all-NaN columns
    df = df.dropna(axis=1, how='all')
    return df


def _remove_duplicate_columns(df):
    """Drop repeated column names, keeping the first occurrence."""
    if not isinstance(df, pd.DataFrame):
        return pd.DataFrame()

    result = df.copy()
    result.columns = result.columns.astype(str).str.strip()
    return result.loc[:, ~result.columns.duplicated(keep="first")]


REQUIRED = {
    "srcs": ["Sample", "Group", "Rb", "Sr", "Zr"],
    "study": ["Sample", "Group", "Rb", "Sr", "Zr"]
}

def check_schema(df, required, name):
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")


def join_sources_locations():
    """Build a joined table from source chemistry and source locations, purely to
    report join coverage to the user. Called by clean_and_validate()."""
    srcs_locs = state.srcs_locs

    if srcs_locs is None or srcs_locs.empty:
        print("⚠️ Source locations not loaded.")
        return

    location_columns = ["Group", "Lat", "Long", "Name"]
    missing_srcs_locs = [c for c in location_columns if c not in srcs_locs.columns]
    if missing_srcs_locs:
        print(f"⚠️ srcs_locs missing columns: {missing_srcs_locs}")
        print(f"Available columns: {srcs_locs.columns.tolist()}")
        return

    srcs_locs_coords = srcs_locs[location_columns].drop_duplicates(subset=["Group"])
    srcs_locs_coords = srcs_locs_coords.rename(columns={"Lat": "Lat_loc", "Long": "Long_loc"})

    joined_srcs = state.srcs.merge(srcs_locs_coords, on='Group', how='left')

    print("✅ Joined source chemistry to source locations.")
    print(f"Rows: {len(joined_srcs)}")

    if not joined_srcs.empty and "Lat_loc" in joined_srcs.columns and "Long_loc" in joined_srcs.columns:
        if joined_srcs[["Lat_loc", "Long_loc"]].isna().any(axis=None):
            print("⚠️ Some rows have missing coordinates after the join.")
        else:
            print("✅ All joined rows have coordinates.")


def clean_and_validate(raw_srcs, raw_locs, raw_study):
    """Clean the uploaded tables, validate schema, and join sources to locations.

    Populates the shared `state` object. Called from the Tab 1 "Load & Continue"
    button so it runs on-demand rather than at notebook load time.
    """
    state.srcs = clean_geochem_df(raw_srcs)
    state.study = clean_geochem_df(raw_study)
    state.srcs_locs = raw_locs.copy()

    # Drop duplicate columns now so downstream code (color map, plots) can
    # safely index state.srcs['Group'] / state.study['Group'] as a Series.
    state.srcs = _remove_duplicate_columns(state.srcs)
    state.study = _remove_duplicate_columns(state.study)

    # Only enforce schema when the required columns are present in the data.
    for df_name, df in [("Sources", state.srcs), ("Study", state.study)]:
        if df is not None and not df.empty:
            try:
                check_schema(df, REQUIRED["srcs" if df_name == "Sources" else "study"], df_name)
            except ValueError as exc:
                print(f"⚠️ {exc}")

    print('Complete Data Types and Study Sample Table used in this visualization:')
    display(state.study.dtypes)
    display(state.study.head())

    join_sources_locations()


In [ ]:
# CELL 4 - SOURCE SELECTION WITH FIGUREWIDGET
# Builds the lasso-selection map (Tab 2) and wires the "Get Selection" button
# to trigger the downstream analysis (Tab 3) automatically.

def _patched_delta_handler(fig_instance):
    """
    Monkey-patch FigureWidget handlers to:
    1. Skip trace deltas missing 'uid'
    2. Filter out invalid property paths like 'mapbox._derived' from relayout events
    3. Guard against Plotly versions where the relayout store is a plain dict instead of a widget
    This allows lasso selection to work without crashing.
    """
    # Patch 1: Handle trace deltas
    original_delta_handler = getattr(fig_instance, '_handler_js2py_traceDeltas', None)
    if original_delta_handler is not None:
        def safe_delta_handler(change):
            try:
                if isinstance(change.get('new'), list):
                    trace_deltas = change['new']
                    deltas_with_uid = [d for d in trace_deltas if isinstance(d, dict) and 'uid' in d]
                    if deltas_with_uid:
                        change_copy = dict(change)
                        change_copy['new'] = deltas_with_uid
                        try:
                            original_delta_handler(change_copy)
                        except (KeyError, IndexError):
                            pass
                else:
                    original_delta_handler(change)
            except Exception:
                pass

        fig_instance._handler_js2py_traceDeltas = safe_delta_handler
        trace_store = getattr(fig_instance, '_traceDeltas', None)
        if trace_store is not None and hasattr(trace_store, 'unobserve') and hasattr(trace_store, 'observe'):
            try:
                trace_store.unobserve(original_delta_handler, names='value')
            except Exception:
                pass
            try:
                trace_store.observe(safe_delta_handler, names='value')
            except Exception:
                pass

    # Patch 2: Handle relayout events to filter invalid property paths
    original_relayout_handler = getattr(fig_instance, '_handler_js2py_relayout', None)
    if original_relayout_handler is not None:
        def safe_relayout_handler(change):
            try:
                relayout_data = change.get('new', {})
                if isinstance(relayout_data, dict):
                    cleaned_relayout = {k: v for k, v in relayout_data.items() if '_derived' not in k}
                    if cleaned_relayout:
                        change_copy = dict(change)
                        change_copy['new'] = cleaned_relayout
                        original_relayout_handler(change_copy)
                else:
                    original_relayout_handler(change)
            except ValueError as e:
                if 'Invalid property path' in str(e):
                    pass
                else:
                    raise
            except Exception:
                pass

        fig_instance._handler_js2py_relayout = safe_relayout_handler
        relayout_store = getattr(fig_instance, '_js2py_relayout', None)
        if relayout_store is not None and hasattr(relayout_store, 'unobserve') and hasattr(relayout_store, 'observe'):
            try:
                relayout_store.unobserve(original_relayout_handler, names='value')
            except Exception:
                pass
            try:
                relayout_store.observe(safe_relayout_handler, names='value')
            except Exception:
                pass


def build_map_selection_ui(host_out):
    """Render Tab 2 (source map + lasso selection) into host_out."""
    with host_out:
        host_out.clear_output(wait=True)
        display(widgets.HTML(
            '<h3>Joined South American obsidian source chemistry with coordinates</h3>'
            '<p>The table above (Tab 1) is joined to the source location coordinates '
            'by <code>Group</code>. Use the lasso tool below to select obsidian '
            'sources from your region of interest, then click "Get Selection" to '
            'update the biplot and ternary plots on the next tab.</p>'
        ))

        srcs_locs = state.srcs_locs.copy()
        if 'Lat' not in srcs_locs.columns or 'Long' not in srcs_locs.columns:
            print("⚠️ srcs_locs is missing Lat/Long columns.")
            return

        srcs_locs['Lat'] = pd.to_numeric(srcs_locs['Lat'], errors='coerce')
        srcs_locs['Long'] = pd.to_numeric(srcs_locs['Long'], errors='coerce')
        srcs_locs = srcs_locs.dropna(subset=['Lat', 'Long'])

        if srcs_locs.empty:
            print("⚠️ srcs_locs contains no valid coordinates after cleaning.")
            return

        names = srcs_locs['Name'].astype(str).tolist()
        groups = srcs_locs['Group'].astype(str).tolist()
        lats = srcs_locs['Lat'].astype(float).tolist()
        lons = srcs_locs['Long'].astype(float).tolist()

        # --- Split sources into joined (have chemistry data) vs unjoined, for distinct symbols ---
        joined_groups_set = (
            set(state.srcs['Group'].dropna().astype(str))
            if isinstance(state.srcs, pd.DataFrame) and not state.srcs.empty and 'Group' in state.srcs.columns
            else set()
        )

        joined_idx = [i for i, g in enumerate(groups) if g in joined_groups_set]
        unjoined_idx = [i for i, g in enumerate(groups) if g not in joined_groups_set]

        joined_names = [names[i] for i in joined_idx]
        joined_groups_list = [groups[i] for i in joined_idx]
        unjoined_names = [names[i] for i in unjoined_idx]
        unjoined_groups_list = [groups[i] for i in unjoined_idx]

        # --- Zoom helper — center on sources with chemistry data when available ---
        center_lats = [lats[i] for i in joined_idx] or lats
        center_lons = [lons[i] for i in joined_idx] or lons
        center_lat = np.mean(center_lats)
        center_lon = np.mean(center_lons)
        max_span = max(max(center_lats) - min(center_lats), max(center_lons) - min(center_lons)) if len(center_lats) > 1 else 1
        zoom = (5 if max_span > 10 else
                6 if max_span > 5 else
                7 if max_span > 2 else
                8 if max_span > 1 else
                9 if max_span > 0.5 else 11)

        # --- Build traces with explicit UIDs; symbol + color distinguish joined vs unjoined sources ---
        trace_joined = go.Scattermapbox(
            lat=[lats[i] for i in joined_idx],
            lon=[lons[i] for i in joined_idx],
            mode='markers+text',
            text=joined_names,
            textposition='top center',
            customdata=joined_groups_list,
            marker=dict(size=10, color='steelblue', symbol='circle'),
            selected=dict(marker=dict(size=14, color='red')),
            unselected=dict(marker=dict(opacity=0.4)),
            hovertemplate='<b>%{text}</b><br><b>Group:</b> %{customdata}<br>Has chemistry data<extra></extra>',
            name='Sources with chemistry data'
        )
        trace_joined.uid = "scattermapbox_joined"

        trace_unjoined = go.Scattermapbox(
            lat=[lats[i] for i in unjoined_idx],
            lon=[lons[i] for i in unjoined_idx],
            mode='markers+text',
            text=unjoined_names,
            textposition='top center',
            textfont=dict(color='lightgray'),
            customdata=unjoined_groups_list,
            marker=dict(size=10, color='black', symbol='x'),
            selected=dict(marker=dict(size=14, color='red')),
            unselected=dict(marker=dict(opacity=0.4)),
            hovertemplate='<b>%{text}</b><br><b>Group:</b> %{customdata}<br>No chemistry data<extra></extra>',
            name='Sources without chemistry data'
        )
        trace_unjoined.uid = "scattermapbox_unjoined"

        # --- Create FigureWidget and apply monkey-patch ---
        fig = FigureWidget(data=[trace_joined, trace_unjoined])
        _patched_delta_handler(fig)

        fig.update_layout(
            mapbox=dict(
                style='open-street-map',
                center=dict(lat=center_lat, lon=center_lon),
                zoom=zoom
            ),
            dragmode='lasso',
            height=500,
            margin=dict(r=0, l=0, t=30, b=0),
            title="🗺️ Obsidian Source Locations — lasso to select, then click Get Selection",
            hovermode='closest',
            showlegend=True
        )

        display(fig)

        selection_out = widgets.Output()
        button = widgets.Button(
            description='Get Selection',
            button_style='primary',
            icon='check',
            layout=widgets.Layout(width='180px'),
        )

        def on_get_selection(b):
            selection_out.clear_output(wait=True)

            with selection_out:
                try:
                    sel_joined = fig.data[0].selectedpoints or [] if len(fig.data) > 0 else []
                    sel_unjoined = fig.data[1].selectedpoints or [] if len(fig.data) > 1 else []

                    if len(sel_joined) == 0 and len(sel_unjoined) == 0:
                        print("⚠️  No points selected. Draw a lasso on the map first.")
                        return

                    selected_names = [joined_names[i] for i in sel_joined] + [unjoined_names[i] for i in sel_unjoined]
                    selected_groups = [joined_groups_list[i] for i in sel_joined] + [unjoined_groups_list[i] for i in sel_unjoined]

                    # Remove duplicates while preserving order
                    seen = set()
                    selected_groups = [g for g in selected_groups if not (g in seen or seen.add(g))]

                    state.selected_names = selected_names
                    state.selected_groups = selected_groups

                    print(f"✅  {len(selected_groups)} source(s) selected:\n")
                    for g in selected_groups:
                        print(f"  • {g}")
                except Exception as e:
                    print(f"❌ Error reading selection: {e}")
                    return

            render_analysis(tab3_out)
            tabs.selected_index = 2

        button.on_click(on_get_selection)
        display(widgets.HTML('<b>After lasso-selecting sources, click:</b>'))
        display(widgets.VBox([button, selection_out]))


def render_selection_summary():
    """Selection summary — invoked by render_analysis() after each "Get Selection" click."""
    if not state.selected_names:
        display(HTML("""
        <div style="padding: 10px; background-color: #fff3e0; border-radius: 5px; border-left: 4px solid #FF9800;">
            <b>⚠️ No selections yet.</b> Click "Get Selection" after selecting sources on the map.
        </div>
        """))
        return

    try:
        display(HTML(f"""
        <div style="padding: 10px; background-color: #c8e6c9; border-radius: 5px; border-left: 4px solid #4CAF50;">
            <b>✅ Success!</b> Selected <b>{len(state.selected_names)}</b> source(s)
        </div>
        """))

        summary = []
        for name, group in zip(state.selected_names, state.selected_groups):
            if group in state.srcs['Group'].values:
                count = len(state.srcs[state.srcs['Group'] == group])
                summary.append({"Source": name, "Group": group, "Count": count})
            else:
                print(f"  ⚠️ Warning: {name} ({group}) not found in srcs data")

        if len(summary) > 0:
            summary_df = pd.DataFrame(summary).sort_values("Count", ascending=False)
            display(summary_df)
        else:
            display(HTML("""
            <div style="padding: 10px; background-color: #ffcdd2; border-radius: 5px; border-left: 4px solid #f44336;">
                <b>❌ Error:</b> No selected sources found in data
            </div>
            """))
    except Exception as e:
        display(HTML(f"""
        <div style="padding: 10px; background-color: #ffcdd2; border-radius: 5px; border-left: 4px solid #f44336;">
            <b>❌ Error processing selection:</b> {str(e)}
        </div>
        """))


In [ ]:
# CELL 5 - APPLY MAP SELECTION AND BUILD COLOR MAP
# Invoked by render_analysis() once a map selection has been applied.

def apply_selection():
    if not state.selected_groups:
        print("❌ Error: No sources selected. Please select sources from the map first.")
        state.srcs_subset = pd.DataFrame()
        state.onesample = pd.DataFrame()
        return

    try:
        srcs_subset = state.srcs[state.srcs['Group'].isin(state.selected_groups)].copy()

        # Prepare a normalized locations table from srcs_locs (handle varied column names)
        def _find_col(df, names):
            names_low = [n.lower() for n in names]
            for c in df.columns:
                if c.lower() in names_low:
                    return c
            return None

        srcs_locs = state.srcs_locs
        group_col = _find_col(srcs_locs, ['Group', 'group', 'Source', 'source']) or 'Group'
        lat_col = _find_col(srcs_locs, ['Lat', 'Latitude', 'lat', 'latitude', 'Lat_loc'])
        long_col = _find_col(srcs_locs, ['Long', 'Longitude', 'Long_loc', 'Lon', 'lon', 'longitude'])
        name_col = _find_col(srcs_locs, ['Name', 'name', 'Source', 'source'])

        if lat_col is None or long_col is None:
            raise KeyError(f"srcs_locs missing coordinate columns. Available columns: {list(srcs_locs.columns)}")

        loc_rename = {group_col: 'Group', lat_col: 'Lat', long_col: 'Long'}
        if name_col:
            loc_rename[name_col] = 'Name'
        srcs_locs_norm = srcs_locs.rename(columns=loc_rename)

        if 'Group' not in srcs_locs_norm.columns:
            raise KeyError(f"Could not locate a 'Group' column in srcs_locs. Available: {list(srcs_locs.columns)}")

        srcs_locs_coords = srcs_locs_norm[['Group', 'Lat', 'Long'] + (['Name'] if 'Name' in srcs_locs_norm.columns else [])]
        srcs_locs_coords = srcs_locs_coords.drop_duplicates(subset=['Group']).reset_index(drop=True)

        srcs_subset = srcs_subset.merge(srcs_locs_coords, on='Group', how='left')

        if len(srcs_subset) == 0:
            raise ValueError(f"No data found for selected sources: {state.selected_groups}")

        if 'Lat' not in srcs_subset.columns or 'Long' not in srcs_subset.columns:
            print(f"⚠️ After merge, location columns missing. Available columns: {list(srcs_subset.columns)}")
        elif srcs_subset[['Lat', 'Long']].isna().any().any():
            print("⚠️ Some selected sources have no matching location data.")

        print(f"✅ Filtered to {len(srcs_subset)} samples from {len(state.selected_groups)} selected sources")

        counts = srcs_subset['Group'].value_counts()
        onesample = srcs_subset[srcs_subset['Group'].map(counts) < 2]
        srcs_subset = srcs_subset[srcs_subset['Group'].map(counts) >= 2]

        print(f"✅ {len(srcs_subset)} samples for ellipses")
        print(f"✅ {len(onesample)} samples for points (< 2 per source)")

        state.srcs_subset = srcs_subset
        state.onesample = onesample
    except ValueError as e:
        print(f"❌ Error: {e}")
        state.srcs_subset = pd.DataFrame()
        state.onesample = pd.DataFrame()
    except KeyError as e:
        print(f"❌ Key error: {e}")
        state.srcs_subset = pd.DataFrame()
        state.onesample = pd.DataFrame()
    except Exception as e:
        print(f"❌ Unexpected error: {type(e).__name__}: {e}")
        state.srcs_subset = pd.DataFrame()
        state.onesample = pd.DataFrame()


def build_color_map():
    """Assign a color to each Source and each Study Group for consistency.
    Invoked by render_analysis() once the selection has been applied."""
    unique_groups = state.srcs['Group'].dropna().unique()
    study_groups = state.study['Group'].dropna().unique() if 'Group' in state.study.columns else []
    all_groups = pd.unique(np.concatenate([unique_groups, study_groups])) if len(study_groups) else unique_groups
    colors = DEFAULT_PLOTLY_COLORS

    # Cycle colors if more groups than colors
    state.name_to_color = {name: colors[i % len(colors)] for i, name in enumerate(all_groups)}


In [ ]:
# CELL 6 - BIPLOT
# Rendered by render_analysis() once a map selection has been applied.

def ellipse_points(x, y, n_std=1.0, n_points=120):
    """Return an (n_points, 2) array tracing the n_std confidence ellipse of x, y.

    Shared by both the biplot and ternary plot so the ellipse math only exists once.
    """
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(pd.Series(y), errors="coerce").to_numpy(dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        return None

    covariance = np.cov(x, y)

    if not np.isfinite(covariance).all():
        return None

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    eigenvalues = np.maximum(eigenvalues, 0)

    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    theta = np.linspace(0, 2 * np.pi, n_points)
    unit_circle = np.column_stack([
        np.cos(theta),
        np.sin(theta)
    ])

    ellipse = (
        unit_circle
        @ np.diag(n_std * np.sqrt(eigenvalues))
        @ eigenvectors.T
    )

    ellipse += [x.mean(), y.mean()]

    return ellipse


def build_biplot(x_col, y_col):
    GROUP_COL = "Group"
    fig = go.Figure()
    study_trace_indices = []

    srcs_subset_valid = (
        state.srcs_subset.copy()
        if isinstance(state.srcs_subset, pd.DataFrame) and not state.srcs_subset.empty
        else pd.DataFrame()
    )

    source_data = (
        srcs_subset_valid.copy()
        if (
            isinstance(srcs_subset_valid, pd.DataFrame)
            and not srcs_subset_valid.empty
            and GROUP_COL in srcs_subset_valid.columns
        )
        else pd.DataFrame()
    )

    # Add source ellipses.
    if (
        not source_data.empty
        and x_col in source_data.columns
        and y_col in source_data.columns
    ):
        source_groups = (
            source_data[GROUP_COL]
            .dropna()
            .astype(str)
            .unique()
        )

        for source_group in source_groups:
            group_data = source_data[
                source_data[GROUP_COL].astype(str) == source_group
            ]

            ellipse = ellipse_points(
                group_data[x_col],
                group_data[y_col],
                n_std=1.0
            )

            if ellipse is None:
                continue

            ellipse_x, ellipse_y = ellipse[:, 0], ellipse[:, 1]
            color = state.name_to_color.get(source_group, "gray")

            fig.add_trace(go.Scatter(
                x=ellipse_x,
                y=ellipse_y,
                mode="lines",
                fill="toself",
                fillcolor=color,
                line=dict(color=color, width=2),
                opacity=0.35,
                name=f"{source_group} Source Ellipse",
                hoverinfo="skip"
            ))

            numeric_x = pd.to_numeric(
                group_data[x_col],
                errors="coerce"
            )

            numeric_y = pd.to_numeric(
                group_data[y_col],
                errors="coerce"
            )

            fig.add_trace(go.Scatter(
                x=[numeric_x.mean()],
                y=[numeric_y.mean()],
                mode="text",
                text=[f"{source_group} Source"],
                textposition="middle center",
                textfont=dict(size=11, color=color),
                showlegend=False,
                hoverinfo="skip"
            ))

    # Add source samples with fewer than two observations.
    onesample = state.onesample
    if (
        isinstance(onesample, pd.DataFrame)
        and not onesample.empty
        and x_col in onesample.columns
        and y_col in onesample.columns
    ):
        valid = (
            pd.to_numeric(onesample[x_col], errors="coerce").notna()
            & pd.to_numeric(onesample[y_col], errors="coerce").notna()
        )

        source_sample_text = (
            onesample.loc[valid, "Sample"]
            if "Sample" in onesample.columns
            else None
        )

        fig.add_trace(go.Scatter(
            x=pd.to_numeric(
                onesample.loc[valid, x_col],
                errors="coerce"
            ),
            y=pd.to_numeric(
                onesample.loc[valid, y_col],
                errors="coerce"
            ),
            name="Source Sample",
            mode="markers",
            marker=dict(symbol="x", size=8, color="black"),
            text=source_sample_text,
            hovertemplate="Source: %{text}<br><extra></extra>",
            showlegend=True
        ))

    # Add study samples grouped by Group.
    study = state.study
    if (
        isinstance(study, pd.DataFrame)
        and not study.empty
        and GROUP_COL in study.columns
        and x_col in study.columns
        and y_col in study.columns
    ):
        study_groups = (
            study[GROUP_COL]
            .dropna()
            .astype(str)
            .unique()
        )

        for study_group in study_groups:
            group_data = study[
                study[GROUP_COL].astype(str) == study_group
            ].copy()

            x_values = pd.to_numeric(
                group_data[x_col],
                errors="coerce"
            )

            y_values = pd.to_numeric(
                group_data[y_col],
                errors="coerce"
            )

            valid = x_values.notna() & y_values.notna()

            if not valid.any():
                continue

            color = state.name_to_color.get(study_group, "gray")

            sample_text = (
                group_data.loc[valid, "Sample"]
                if "Sample" in group_data.columns
                else None
            )

            study_trace_indices.append(len(fig.data))

            fig.add_trace(go.Scatter(
                x=x_values.loc[valid],
                y=y_values.loc[valid],
                name=f"Study: {study_group}",
                mode="markers",
                text=sample_text,
                hovertemplate="Sample: %{text}<br><extra></extra>",
                marker=dict(
                    size=8,
                    symbol="circle",
                    color=color
                ),
                showlegend=True
            ))

    fig.update_layout(
        title=dict(
            text="Connecting Study Samples with known obsidian sources",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title=x_col,
        yaxis_title=y_col,
        height=600,
        margin=dict(
            l=0,
            r=240,
            t=120,
            b=0
        ),
        legend=dict(
            x=1.02,
            y=0.98,
            xanchor="left",
            yanchor="top"
        ),
        updatemenus=[
            dict(
                type="buttons",
                direction="down",
                x=1.02,
                y=1.08,
                xanchor="left",
                yanchor="bottom",
                buttons=[
                    dict(
                        label="Labels OFF",
                        method="restyle",
                        args=[
                            {"mode": "markers"},
                            study_trace_indices
                        ]
                    ),
                    dict(
                        label="Labels ON",
                        method="restyle",
                        args=[
                            {
                                "mode": "markers+text",
                                "textposition": "top center"
                            },
                            study_trace_indices
                        ]
                    )
                ]
            )
        ]
    )

    return fig


def render_biplot_ui(host_out):
    """Render the biplot section (element dropdowns + plot) into host_out."""
    state.srcs = _remove_duplicate_columns(state.srcs)
    state.study = _remove_duplicate_columns(state.study)
    state.srcs_subset = _remove_duplicate_columns(state.srcs_subset)

    plot_elements = ["Rb", "Sr", "Zr", "Ba", "Nb", "Y", "Th", "U"]
    available_elements = [
        col for col in plot_elements
        if col in state.srcs.columns and col in state.study.columns
    ]

    with host_out:
        display(HTML(
            "<h3>Biplot</h3>"
            "<p style='color:grey;font-size:12px;'>You may zoom on the graph, "
            "change the element variables on the axes, enable label display, "
            "and export the figure.</p>"
        ))

        if not available_elements:
            print("⚠️ No common geochemical columns are available in srcs and study.")
            return

        x_default = "Sr" if "Sr" in available_elements else available_elements[0]
        y_default = "Rb" if "Rb" in available_elements else available_elements[0]

        x_menu = widgets.Dropdown(
            options=available_elements,
            value=x_default,
            description="X axis:",
            style={"description_width": "initial"}
        )

        y_menu = widgets.Dropdown(
            options=available_elements,
            value=y_default,
            description="Y axis:",
            style={"description_width": "initial"}
        )

        display(widgets.HBox([x_menu, y_menu]))

        biplot_output = widgets.Output()

        def update_biplot(change=None):
            with biplot_output:
                biplot_output.clear_output(wait=True)
                display(build_biplot(x_menu.value, y_menu.value))

        x_menu.observe(update_biplot, names="value")
        y_menu.observe(update_biplot, names="value")

        display(biplot_output)
        update_biplot()


In [ ]:
# CELL 7 - TERNARY PLOT + ANALYSIS ORCHESTRATOR
# Rendered by render_analysis() once a map selection has been applied.

def normalize_composition(df, cols):
    values = df[cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    totals = np.sum(values, axis=1, keepdims=True)

    valid = np.isfinite(totals[:, 0]) & (totals[:, 0] > 0)
    result = np.full_like(values, np.nan)

    with np.errstate(divide="ignore", invalid="ignore"):
        result[valid] = values[valid] / totals[valid]

    return result


def build_ternary_plot(a_col, b_col, c_col):
    GROUP_COL = "Group"
    cols = [a_col, b_col, c_col]
    fig = go.Figure()
    study_trace_indices = []

    srcs_subset_valid = (
        state.srcs_subset.copy()
        if isinstance(state.srcs_subset, pd.DataFrame) and not state.srcs_subset.empty
        else pd.DataFrame()
    )
    study = state.study

    source_data = srcs_subset_valid.copy()
    source_fraction = normalize_composition(source_data, cols) if not source_data.empty else None
    study_fraction = normalize_composition(study, cols) if not study.empty else None

    # Source ellipses
    if source_fraction is not None and not source_data.empty and GROUP_COL in source_data.columns:
        for source_group in source_data[GROUP_COL].dropna().unique():
            group_mask = (
                source_data[GROUP_COL].astype(str)
                == str(source_group)
            ).to_numpy()

            data = source_fraction[group_mask]
            valid = np.isfinite(data).all(axis=1)
            data = data[valid]

            if len(data) < 2:
                continue

            ellipse = ellipse_points(
                data[:, 0],
                data[:, 1],
                n_std=1.0
            )

            if ellipse is None:
                continue

            a_values = ellipse[:, 0]
            b_values = ellipse[:, 1]
            c_values = 1 - a_values - b_values

            color = state.name_to_color.get(source_group, "gray")

            fig.add_trace(go.Scatterternary(
                a=a_values,
                b=b_values,
                c=c_values,
                mode="lines",
                line=dict(color=color),
                fill="toself",
                opacity=0.35,
                name=f"{source_group} Source"
            ))

    # Study samples
    if study_fraction is not None and not study.empty and GROUP_COL in study.columns:
        for study_group in study[GROUP_COL].dropna().unique():
            group_mask = (
                study[GROUP_COL].astype(str)
                == str(study_group)
            ).to_numpy()

            data = study_fraction[group_mask]
            valid = np.isfinite(data).all(axis=1)
            data = data[valid]

            if len(data) == 0:
                continue

            color = state.name_to_color.get(study_group, "gray")
            trace_index = len(fig.data)

            point_labels = (
                study.loc[group_mask, "Name"].to_numpy()[valid]
                if "Name" in study.columns
                else study.loc[group_mask, "Sample"].to_numpy()[valid]
                if "Sample" in study.columns
                else [str(study_group)] * len(data)
            )

            fig.add_trace(go.Scatterternary(
                a=data[:, 0],
                b=data[:, 1],
                c=data[:, 2],
                mode="markers",
                name=f"Study: {study_group}",
                text=point_labels,
                hovertemplate="Name: %{text}<extra></extra>",
                marker=dict(size=8, color=color),
                showlegend=True
            ))

            study_trace_indices.append(trace_index)

    fig.update_layout(
        ternary=dict(
            sum=1,
            aaxis=dict(title=a_col),
            baxis=dict(title=b_col),
            caxis=dict(title=c_col)
        ),
        title=dict(
            text=f"Ternary Plot: {a_col}, {b_col}, {c_col}",
            x=0.5,
            xanchor="center"
        ),
        legend=dict(
            x=1.02,
            y=1,
            xanchor="left",
            yanchor="top"
        ),
        height=550,
        updatemenus=[
            dict(
                type="buttons",
                direction="down",
                x=1.02,
                y=1.16,
                xanchor="left",
                yanchor="bottom",
                buttons=[
                    dict(
                        label="Labels OFF",
                        method="restyle",
                        args=[{"mode": "markers"}, study_trace_indices]
                    ),
                    dict(
                        label="Labels ON",
                        method="restyle",
                        args=[
                            {
                                "mode": "markers+text",
                                "textposition": "top center"
                            },
                            study_trace_indices
                        ]
                    )
                ]
            )
        ]
    )

    return fig


def render_ternary_ui(host_out):
    """Render the ternary plot section (element dropdowns + plot) into host_out."""
    plot_elements = ["Rb", "Sr", "Zr", "Ba", "Nb", "Y", "Th", "U"]
    ternary_elements = [
        column for column in plot_elements
        if column in state.srcs.columns and column in state.study.columns
    ]

    with host_out:
        display(HTML(
            "<h3>Ternary Plot</h3>"
            "<p style='color:grey;font-size:12px;'>Examine relationship between "
            "samples and the ellipses representing known sources. Zoom into the "
            "ternary diagram and change the variables shown above if you wish to "
            "view other elements in the diagram.</p>"
        ))

        if len(ternary_elements) < 3:
            print("⚠️ Fewer than three common geochemical columns are available in srcs and study.")
            return

        a_default = "Rb" if "Rb" in ternary_elements else ternary_elements[0]
        b_default = "Sr" if "Sr" in ternary_elements else ternary_elements[1]
        c_default = "Zr" if "Zr" in ternary_elements else ternary_elements[2]

        ternary_a_menu = widgets.Dropdown(options=ternary_elements, value=a_default, description="A axis:")
        ternary_b_menu = widgets.Dropdown(options=ternary_elements, value=b_default, description="B axis:")
        ternary_c_menu = widgets.Dropdown(options=ternary_elements, value=c_default, description="C axis:")

        display(widgets.HBox([ternary_a_menu, ternary_b_menu, ternary_c_menu]))

        ternary_output = widgets.Output()

        def update_ternary(change=None):
            with ternary_output:
                ternary_output.clear_output(wait=True)
                display(build_ternary_plot(
                    ternary_a_menu.value,
                    ternary_b_menu.value,
                    ternary_c_menu.value
                ))

        ternary_a_menu.observe(update_ternary, names="value")
        ternary_b_menu.observe(update_ternary, names="value")
        ternary_c_menu.observe(update_ternary, names="value")

        display(ternary_output)
        update_ternary()


def render_analysis(host_out):
    """Orchestrates Tab 3: selection summary, biplot, and ternary plot side by side.

    Invoked by the "Get Selection" button callback in build_map_selection_ui().
    """
    with host_out:
        host_out.clear_output(wait=True)
        display(widgets.HTML(
            '<h3>Step 3 — Analysis updates automatically</h3>'
            '<p>The summary, biplot, and ternary plot below reflect the sources '
            'you lasso-selected on the Map Selection tab. Mouse over the points '
            'to view the unique ID numbers of samples and note their relationship '
            'to ellipses for known obsidian sources.</p>'
        ))
        render_selection_summary()

    apply_selection()
    build_color_map()

    biplot_out = widgets.Output()
    ternary_out = widgets.Output()
    with host_out:
        display(widgets.HBox(
            [biplot_out, ternary_out],
            layout=widgets.Layout(width="100%", flex_flow="row wrap", justify_content="space-between")
        ))

    render_biplot_ui(biplot_out)
    render_ternary_ui(ternary_out)


In [ ]:
# CELL 8 - TAB 1: DATA UPLOAD
# Builds the "Data Upload" tab: default local CSVs are attempted first, and
# the user can override any of the three tables by uploading a CSV or
# providing a Google Sheets / direct CSV URL, then clicking "Load & Continue".

def _url_loader_row(title, on_load):
    """A labeled Text box + Load button used for the per-table URL option."""
    url_text = widgets.Text(
        placeholder="Paste a Google Sheets link or direct CSV URL",
        layout=widgets.Layout(width="420px")
    )
    load_btn = widgets.Button(description="Load from URL", icon="link", layout=widgets.Layout(width="140px"))
    url_out = widgets.Output()

    def _on_click(b):
        with url_out:
            url_out.clear_output(wait=True)
            if not url_text.value.strip():
                print("⚠️ Paste a URL first.")
                return
            try:
                on_load(gsheet_to_csv_url(url_text.value))
            except Exception as e:
                print(f"❌ Error loading from URL: {e}")

    load_btn.on_click(_on_click)
    display(widgets.HTML(f"<b>{title}</b>"))
    display(widgets.HBox([url_text, load_btn]), url_out)


def build_data_upload_ui(host_out):
    """Render Tab 1 (Data Upload) into host_out."""
    with host_out:
        host_out.clear_output(wait=True)

        display(widgets.HTML("""
        <h3>Step 1 — Provide the three data tables</h3>
        <ul style="font-size:13px;">
          <li><b>study_samples.csv</b> — geochemistry of the study samples for the current investigation.</li>
          <li><b>South_Am_sources_southern_XRF.csv</b> — obsidian source geochemistry.</li>
          <li><b>South_Am_sources_all_locations.csv</b> — obsidian source coordinates.</li>
        </ul>
        <p style="font-size:13px;">Local defaults bundled with this notebook are loaded automatically below.
        To use your own data, either upload a CSV file or paste a link to a published
        Google Sheet (or any direct CSV URL) for each table, then click
        <b>"Load & Continue"</b>.</p>
        """))

        status_out = widgets.Output()
        with status_out:
            for message in load_default_local_files():
                print(message)
        display(status_out)

        display(widgets.HTML("<hr>"))

        # --- 1. Sources Chemistry (upload or URL) ---
        display(widgets.HTML("<h4>1. Sources Chemistry (XRF Data)</h4>"))
        chem_upload = widgets.FileUpload(accept=".csv", multiple=False, description="Select CSV")
        chem_out = widgets.Output()

        def on_chem_change(change):
            uploaded = _first_upload(chem_upload)
            if uploaded is None:
                return
            with chem_out:
                chem_out.clear_output(wait=True)
                try:
                    _raw.srcs = read_csv_with_encodings(uploaded.get("content"))
                    print(f"✅ Loaded Sources Chemistry: {uploaded.get('name')} ({len(_raw.srcs)} rows, {len(_raw.srcs.columns)} cols)")
                except Exception as e:
                    print(f"❌ Error: {e}")

        chem_upload.observe(on_chem_change, names="value")
        display(widgets.HBox([chem_upload]), chem_out)

        def _load_chem_url(url):
            _raw.srcs = read_csv_with_encodings(url)
            print(f"✅ Loaded Sources Chemistry from URL ({len(_raw.srcs)} rows, {len(_raw.srcs.columns)} cols)")

        _url_loader_row("...or load Sources Chemistry from a URL", _load_chem_url)

        display(widgets.HTML("<hr>"))

        # --- 2. Sources Locations (upload or URL) ---
        display(widgets.HTML("<h4>2. Sources Locations (Coordinates)</h4>"))
        loc_upload = widgets.FileUpload(accept=".csv", multiple=False, description="Select CSV")
        loc_out = widgets.Output()

        def on_loc_change(change):
            uploaded = _first_upload(loc_upload)
            if uploaded is None:
                return
            with loc_out:
                loc_out.clear_output(wait=True)
                try:
                    _raw.srcs_locs = _clean_locs(read_csv_with_encodings(uploaded.get("content")))
                    print(f"✅ Loaded Sources Locations: {uploaded.get('name')} ({len(_raw.srcs_locs)} rows)")
                except Exception as e:
                    print(f"❌ Error: {e}")

        loc_upload.observe(on_loc_change, names="value")
        display(widgets.HBox([loc_upload]), loc_out)

        def _load_loc_url(url):
            _raw.srcs_locs = _clean_locs(read_csv_with_encodings(url))
            print(f"✅ Loaded Sources Locations from URL ({len(_raw.srcs_locs)} rows)")

        _url_loader_row("...or load Sources Locations from a URL", _load_loc_url)

        display(widgets.HTML("<hr>"))

        # --- 3. Study Samples (upload or URL) ---
        display(widgets.HTML("<h4>3. Study Samples (Unknowns)</h4>"))
        study_upload = widgets.FileUpload(accept=".csv", multiple=False, description="Select CSV")
        study_out = widgets.Output()

        def on_study_change(change):
            uploaded = _first_upload(study_upload)
            if uploaded is None:
                return
            with study_out:
                study_out.clear_output(wait=True)
                try:
                    _raw.study = _clean_study(read_csv_with_encodings(uploaded.get("content")))
                    print(f"✅ Loaded Study Samples: {uploaded.get('name')} ({len(_raw.study)} rows)")
                except Exception as e:
                    print(f"❌ Error: {e}")

        study_upload.observe(on_study_change, names="value")
        display(widgets.HBox([study_upload]), study_out)

        def _load_study_url(url):
            _raw.study = _clean_study(read_csv_with_encodings(url))
            print(f"✅ Loaded Study Samples from URL ({len(_raw.study)} rows)")

        _url_loader_row("...or load Study Samples from a URL", _load_study_url)

        display(widgets.HTML("<hr>"))

        # --- Continue button: triggers cleaning, joining, and moves to Tab 2 ---
        continue_out = widgets.Output()
        continue_btn = widgets.Button(
            description="Load & Continue ▶",
            button_style="primary",
            icon="arrow-right",
            layout=widgets.Layout(width="200px")
        )

        def on_continue_click(b):
            with continue_out:
                continue_out.clear_output(wait=True)
                if _raw.srcs.empty or _raw.srcs_locs.empty or _raw.study.empty:
                    print("⚠️ Please provide all three CSV tables (via default local files, uploads, or URLs above) before continuing.")
                    return
                clean_and_validate(_raw.srcs, _raw.srcs_locs, _raw.study)
                print("✅ Data loaded, cleaned, and joined. Opening the Map Selection tab…")
            build_map_selection_ui(tab2_out)
            tabs.selected_index = 1

        continue_btn.on_click(on_continue_click)
        display(continue_btn, continue_out)


In [ ]:
# CELL 9 - APPLICATION TABS
# Defined last so every callback dependency above is guaranteed to exist
# before Voilà/Binder renders the interactive widgets.

tab1_out = widgets.Output()
tab2_out = widgets.Output()
tab3_out = widgets.Output()

tabs = widgets.Tab(children=[tab1_out, tab2_out, tab3_out])
tabs.set_title(0, "1. Data Upload")
tabs.set_title(1, "2. Map Selection")
tabs.set_title(2, "3. Biplot & Ternary")

with tab2_out:
    display(widgets.HTML(
        "<p style='color:grey;'>Complete Tab 1 (Data Upload) and click "
        "\u201cLoad & Continue\u201d to display the source map here.</p>"
    ))
with tab3_out:
    display(widgets.HTML(
        "<p style='color:grey;'>Lasso-select sources on Tab 2 (Map Selection) and click "
        "\u201cGet Selection\u201d to display the biplot and ternary plot here.</p>"
    ))

build_data_upload_ui(tab1_out)

display(tabs)


### The analysis notebook has concluded.
In your data exploration, mouse over the points in the biplot and ternary plot to view the unique ID numbers of those samples and note their relationship to ellipses for known obsidian sources.
